# EGATUR / INE: gasto turistico en Espana

**Que responde este cuaderno**

1. Como evoluciona el gasto total mes a mes y cuanto varia interanualmente.
2. Que paises de residencia aportan mas gasto y con que cuota.
3. Que comunidades autonomas concentran ese gasto.
4. Como se reparte el gasto por partidas.

**Datos**: `data/processed/egatur_master_long.csv` (2.881 filas en formato largo: una fila por tabla, periodo, dimension e indicador). Se descarga por URL al ejecutar.

**Como correrlo**: Runtime > Run all. Solo pandas y matplotlib.

**Unidades**: el CSV trae el valor tal como lo publica el INE, sin columna de escala. Por eso el analisis trabaja con cuotas y variaciones, no con absolutos convertidos.

In [ ]:
import io, pathlib, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/Nicolenki7/ETL_Egatur_INE_Esp/main/'
RUTA = 'data/processed/egatur_master_long.csv'
pathlib.Path('outputs').mkdir(exist_ok=True)

with urllib.request.urlopen(BASE + RUTA) as r:
    d = pd.read_csv(io.BytesIO(r.read()))

d['fecha'] = pd.to_datetime(d['periodo_parsed'])

print('filas:', len(d), '| periodos:', d['periodo_parsed'].nunique())
print('periodo:', d['periodo_parsed'].min(), 'a', d['periodo_parsed'].max())
print()
print(d.groupby('table_name').size().to_string())

## 1. Gasto total por mes

La tabla 10838 trae, ademas de cada pais, una fila `Total` por periodo: esa es la serie agregada y la que usamos para la evolucion.

In [ ]:
total = (d[(d.table_id == 10838) & (d.dimension_value == 'Total')]
         .drop_duplicates('periodo_parsed')
         .sort_values('fecha'))

ultimos12 = total.tail(12)['valor'].sum()
previos12 = total.iloc[-24:-12]['valor'].sum()
var = (ultimos12 / previos12 - 1) * 100 if previos12 else float('nan')
ultimo_periodo = total['periodo_parsed'].iloc[-1]
ultimo_valor = total['valor'].iloc[-1]

print(f'RESPUESTA: en {ultimo_periodo} el gasto total fue {ultimo_valor}, y en los ultimos 12 meses varia {var:.1f} % frente a los 12 anteriores.')

ax = total.tail(60).set_index('fecha')['valor'].plot(figsize=(9, 3.5))
ax.set_ylabel('gasto total (unidad del INE)')
ax.set_title('Gasto turistico total por mes (ultimos 5 anos)')
plt.tight_layout()
plt.savefig('outputs/gasto_mensual.png', dpi=150)
plt.show()

## 2. Que paises sostienen el gasto

Excluimos la fila `Total` para no contar dos veces el agregado y comparamos el ultimo ano movil disponible.

In [ ]:
inicio = total['periodo_parsed'].iloc[-12]
corte = total['periodo_parsed'].iloc[-1]

paises = d[(d.table_id == 10838) & (d.dimension_value != 'Total') &
           (d.periodo_parsed >= inicio) & (d.periodo_parsed <= corte)]

por_pais = paises.groupby('dimension_value')['valor'].sum().sort_values(ascending=False)
cuota = (por_pais / por_pais.sum() * 100).round(1)
print(pd.DataFrame({'gasto': por_pais.round(1), 'cuota_pct': cuota}).head(8).to_string())

lista = ', '.join(cuota.head(3).index)
suma3 = cuota.head(3).sum()
print()
print(f'RESPUESTA: {lista} aportan juntos el {suma3:.1f} % del gasto del ultimo ano.')

ax = cuota.head(6).sort_values().plot(kind='barh', figsize=(7, 3.5))
ax.set_xlabel('% del gasto (ultimos 12 meses)')
ax.set_title('Cuota por pais de residencia')
plt.tight_layout()
plt.savefig('outputs/cuota_pais.png', dpi=150)
plt.show()

## 3. Destino: comunidades autonomas

Misma logica con la tabla 10839 (CCAA destino).

In [ ]:
ccaa = d[(d.table_id == 10839) & (d.dimension_value != 'Total') &
          (d.periodo_parsed >= inicio) & (d.periodo_parsed <= corte)]

por_ccaa = ccaa.groupby('dimension_value')['valor'].sum().sort_values(ascending=False)
cuota_ccaa = (por_ccaa / por_ccaa.sum() * 100).round(1)
print(pd.DataFrame({'gasto': por_ccaa.round(1), 'cuota_pct': cuota_ccaa}).head(8).to_string())

lider = cuota_ccaa.index[0]
print()
print(f'RESPUESTA: {lider} lidera con el {cuota_ccaa.iloc[0]} % del gasto y las tres primeras suman el {cuota_ccaa.head(3).sum():.1f} %.')

## 4. En que se gasta

La tabla 13938 desglosa las partidas de gasto (alojamiento, transporte, etc.).

In [ ]:
partidas = d[(d.table_id == 13938) & (d.periodo_parsed >= inicio) & (d.periodo_parsed <= corte)]

por_partida = partidas.groupby('dimension_value')['valor'].sum().sort_values(ascending=False)
cuota_partida = (por_partida / por_partida.sum() * 100).round(1)
print(pd.DataFrame({'gasto': por_partida.round(1), 'cuota_pct': cuota_partida}).to_string())

mayor = cuota_partida.index[0]
print()
print(f'RESPUESTA: {mayor} es la mayor partida, con el {cuota_partida.iloc[0]} % del gasto.')

ax = cuota_partida.sort_values().plot(kind='barh', figsize=(7, 3.5))
ax.set_xlabel('% del gasto')
ax.set_title('Reparto por partida (ultimos 12 meses)')
plt.tight_layout()
plt.savefig('outputs/partidas.png', dpi=150)
plt.show()

## Limites

- El CSV es el maestro del proyecto: no consulta el INE en vivo.
- `tipo_dato` es 'Dato base' en todas las filas; no hay series corregidas ni desestacionalizadas.
- La unidad no viene en el archivo, asi que no se convierten absolutos: se leen cuotas y variaciones.
- Algunos 'paises' son agrupaciones del INE (por ejemplo Paises Nordicos).